# Verkeersongevallen in België — Exploratory Data Analysis
**Databron:** Statbel — https://statbel.fgov.be/nl/themas/mobiliteit/verkeer-en-vervoer/verkeersongevallen

## Doel
In dit notebook laden we de ruwe data in, verkennen we de structuur, cleanen we de data en beantwoorden we eerste business vragen via visualisaties.

## 1. Imports & configuratie

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Stijl
sns.set_theme(style='whitegrid', palette='Blues_d')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.family'] = 'sans-serif'

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

print('Libraries geladen ✓')

## 2. Data inladen

Download de bestanden van:
https://statbel.fgov.be/nl/themas/mobiliteit/verkeer-en-vervoer/verkeersongevallen

Sla de Excel/CSV bestanden op in `../data/raw/`

In [ ]:
# Pas het pad aan naar jouw bestandsnaam
df = pd.read_excel('../data/raw/TF_ACCIDENTS_2023.xlsx')

print(f'Aantal rijen:    {df.shape[0]:,}')
print(f'Aantal kolommen: {df.shape[1]}')
df.head()

## 3. Data verkenning

In [ ]:
# Kolomtypes en nulls
df.info()

In [ ]:
# Missende waarden per kolom
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print('Kolommen met missende waarden:')
print(missing)

In [ ]:
# Basisstatistieken numerieke kolommen
df.describe()

## 4. Data cleaning

In [ ]:
# Kolomnamen standaardiseren
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')

# Datum kolom naar datetime
# Pas de kolomnaam aan indien nodig
date_col = [c for c in df.columns if 'date' in c or 'datum' in c][0]
df[date_col] = pd.to_datetime(df[date_col], errors='coerce')

# Tijdkolommen aanmaken
df['jaar']  = df[date_col].dt.year
df['maand'] = df[date_col].dt.month
df['dag']   = df[date_col].dt.day_name()
df['uur']   = df[date_col].dt.hour

print('Cleaning klaar ✓')
df.head()

In [ ]:
# Gecleande data opslaan
df.to_csv('../data/processed/ongevallen_clean.csv', index=False)
print('Opgeslagen naar data/processed/ongevallen_clean.csv ✓')

## 5. Analyse — Business vragen

### Vraag 1: Hoe evolueert het aantal ongevallen per jaar?

In [ ]:
per_jaar = df.groupby('jaar').size().reset_index(name='aantal_ongevallen')

fig, ax = plt.subplots()
ax.bar(per_jaar['jaar'], per_jaar['aantal_ongevallen'], color='#1a73e8')
ax.set_title('Aantal verkeersongevallen per jaar', fontsize=14, fontweight='bold')
ax.set_xlabel('Jaar')
ax.set_ylabel('Aantal ongevallen')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.savefig('../data/processed/fig_per_jaar.png', dpi=150)
plt.show()

### Vraag 2: Op welk uur van de dag gebeuren de meeste ongevallen?

In [ ]:
per_uur = df.groupby('uur').size().reset_index(name='aantal')

fig, ax = plt.subplots()
ax.plot(per_uur['uur'], per_uur['aantal'], marker='o', color='#1a73e8', linewidth=2)
ax.fill_between(per_uur['uur'], per_uur['aantal'], alpha=0.15, color='#1a73e8')
ax.set_title('Verkeersongevallen per uur van de dag', fontsize=14, fontweight='bold')
ax.set_xlabel('Uur')
ax.set_ylabel('Aantal ongevallen')
ax.set_xticks(range(0, 24))
plt.tight_layout()
plt.savefig('../data/processed/fig_per_uur.png', dpi=150)
plt.show()

### Vraag 3: Op welke dag van de week?

In [ ]:
dag_volgorde = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
dag_labels   = ['Maandag','Dinsdag','Woensdag','Donderdag','Vrijdag','Zaterdag','Zondag']

per_dag = (df.groupby('dag').size()
             .reindex(dag_volgorde)
             .reset_index(name='aantal'))
per_dag['dag_nl'] = dag_labels

fig, ax = plt.subplots()
colors = ['#1a73e8' if d not in ['Saturday','Sunday'] else '#f6a623' for d in per_dag['dag']]
ax.bar(per_dag['dag_nl'], per_dag['aantal'], color=colors)
ax.set_title('Verkeersongevallen per dag van de week', fontsize=14, fontweight='bold')
ax.set_xlabel('Dag')
ax.set_ylabel('Aantal ongevallen')
plt.tight_layout()
plt.savefig('../data/processed/fig_per_dag.png', dpi=150)
plt.show()

### Vraag 4: Welke provincies hebben de meeste ongevallen?

In [ ]:
# Pas de kolomnaam aan naar jouw dataset
provincie_col = [c for c in df.columns if 'prov' in c][0]

per_provincie = (df.groupby(provincie_col).size()
                   .sort_values(ascending=True)
                   .reset_index(name='aantal'))

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(per_provincie[provincie_col], per_provincie['aantal'], color='#1a73e8')
ax.set_title('Verkeersongevallen per provincie', fontsize=14, fontweight='bold')
ax.set_xlabel('Aantal ongevallen')
plt.tight_layout()
plt.savefig('../data/processed/fig_per_provincie.png', dpi=150)
plt.show()

## 6. Conclusies

> Vul hier jouw bevindingen in na het uitvoeren van de analyse:
> 
> - De piekuren voor verkeersongevallen zijn ...
> - De gevaarlijkste provincie is ...
> - Het aantal ongevallen over de jaren toont ...
> - Vrijdag is de gevaarlijkste dag / het weekend is rustiger omdat ...